# CR-1 候補の一覧（段2 の固定の準備・CPU・API なし）

CR-1 の G0 の前に固定する「段2」（`notes/cr1_lock_pre_g0.md`）に書く、**除外の連鎖のハッシュ**と**候補の一覧のハッシュ**を作ります。

- 問題の中身はどこにも表示・保存しません（ハッシュ・数・問題の ID と番号だけ）。
- これまでの研究（Q3-A〜DP7）の問題の選び方が、それぞれの実行の記録（`RUN_IDENTITY.json`）と同じ問題を選び直せることを確かめてから作ります。
- API は使いません。費用はかかりません。

**使い方**
1. ランタイムは **CPU** で大丈夫です（GPU はいりません）。
2. Colab の「シークレット」に `GITHUB_TOKEN`（いつもの読むだけのもの）。
3. セル1、セル2の順に実行（数分）。結果は Drive の `DECISION_POINTS/cr1/CR1_CANDIDATES_PRE_G0.json` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact code head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, hashlib, time, datetime, platform

drive.mount("/content/drive")
EXPECTED_HEAD = "89e5448eaa93a49456ea84afcff65d03580c44e4"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
OUT = THEME / "cr1"
OUT.mkdir(parents=True, exist_ok=True)
QP = Path("/content/drive/MyDrive/ULTIMAPORK/quality_pork")   # READ-ONLY
RUNS = [("q3a", QP / "q3a_hightemp" / "QUALITY_PORK_Q3A_20261002"), ("q3b", QP / "q3b_topk" / "QUALITY_PORK_Q3B_20261002"),
        ("q4a", QP / "q4a_fence" / "QUALITY_PORK_Q4A_20261003"), ("q4b", QP / "q4b_sc" / "QUALITY_PORK_Q4B_T1_20261003"),
        ("q5", QP / "q5" / "QUALITY_PORK_Q5_A_20261004"), ("q6s", QP / "q6s" / "QUALITY_PORK_Q6S_A_20261004"),
        ("q8t", QP / "q8t" / "QUALITY_PORK_Q8T_A_20261005"),
        ("dp1", THEME / "dp1" / "DP1_1B_A_20261005"), ("dp2", THEME / "dp2" / "DP2_1B_A_20261006"),
        ("dp3", THEME / "dp3" / "DP3_1B_A_20261007"), ("dp4", THEME / "dp4" / "DP4_1B_A_20261008"),
        ("dp5", THEME / "dp5" / "DP5_GEMMA_A_20261008"), ("dp7", THEME / "dp7" / "DP7_GEMMA_A_20261010")]
_missing = [n for n, p in RUNS if not (p / "RUN_IDENTITY.json").exists() and n != "q8t"]
assert not _missing, ("これまでの研究の実行フォルダが見つかりません", _missing)

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN または GH_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
print("exact head:", head)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1 候補の一覧（段2の準備・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1 candidates', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — rows (pinned revision), earlier studies reproduced, chain and candidates; hashes only.
import numpy as np
from pork import dp7_matched as d7
from pork import cr1_items as CI
from pork import cr1_analysis as A

rows, DATA_ID = d7.d2.load_rows_from_hub()
IDS = {name: json.loads((p / "RUN_IDENTITY.json").read_text()) for name, p in RUNS if (p / "RUN_IDENTITY.json").exists()}
assert all(DATA_ID["dataset_revision"] == v["dataset_revision"] for v in IDS.values()), "データの版がこれまでと違います"
mods = {"q3a": d7.q3a, "q3b": d7.q3b, "q4a": d7.q4a, "q4b": d7.q4b, "q5": d7.q5, "q6s": d7.q6, "q8t": d7.q8,
        "dp1": d7.dp, "dp2": d7.d2, "dp3": d7.d3, "dp4": d7.d4, "dp5": d7.d5, "dp7": d7}
earlier = {name: mods[name].select_items(rows) for name in IDS}
for name in IDS:
    assert mods[name].items_sha256(earlier[name]) == IDS[name]["items_sha256"], f"{name} の問題を選び直せませんでした"
chain = CI.chain_and_candidates(rows)
cands = [it for c in chain for it in c["candidates"]]
key = lambda it: (int(it["id"]), str(it["instance"]))
excluded = {e for c in chain for e in c["excluded"]}
overlap = {name: len({key(i) for i in its} & {key(i) for i in cands}) for name, its in earlier.items()}
inside = {name: all(key(i) in excluded for i in its) for name, its in earlier.items()}
assert all(v == 0 for v in overlap.values()), ("候補がこれまでの問題と重なりました", overlap)
assert all(inside.values()), ("これまでの問題が除外の連鎖に入っていません", inside)
summary = {
    "schema": "cr1-candidates-pre-g0-v1", "code_head": head, "utc": datetime.datetime.utcnow().isoformat() + "Z",
    "dataset": DATA_ID, "python": platform.python_version(), "numpy": np.__version__,
    "earlier_runs_reproduced": sorted(IDS), "q8t_identity_found": "q8t" in IDS,
    "n_templates": len(chain), "n_candidates": len(cands),
    "candidates_per_template": sorted({len(c["candidates"]) for c in chain}),
    "n_excluded": len(excluded),
    "exclusion_chain_sha256": CI.chain_sha256(chain),
    "candidates_sha256": CI.candidates_sha256(chain),
    "candidate_ids_sha256": CI.candidate_ids_sha256(chain),
    "candidate_ids": [[c["template"], [[int(it["id"]), str(it["instance"])] for it in c["candidates"]]] for c in chain],
    "seed_item": A.SEED_ITEM,
}
path = OUT / "CR1_CANDIDATES_PRE_G0.json"
path.write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print("書きました:", path, path.stat().st_size, "bytes")
print({k: summary[k] for k in ("n_templates", "n_candidates", "n_excluded", "exclusion_chain_sha256", "candidates_sha256")})
notify('終わりました：候補の一覧のハッシュを Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()